# Computer Exercise 15.7 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: 15.7 Trust-Region and Clipped Surrogate Methods — *TRPO/PPO 하이퍼파라미터*
> **풀이 언어**: Python (NumPy, pandas, Matplotlib)
> **작성 일자**: 2026-07-22

---


## 1. 문제 (원문)

> **3.** Continuing from Problem 2, study the trade-off in the clipping parameter and compare fixed clipping with an adaptive KL-penalty scheme reminiscent of TRPO. (a) Sweep $\varepsilon \in \{0.05, 0.1, 0.2, 0.4\}$ for PPO with the same $(\alpha, K, \text{batch size})$ as in Problem 2, and report learning speed, seed variance, and the resulting effective KL. (b) Implement the *adaptive KL-penalty* objective
$L^{\text{KLPEN}}(\theta) = \mathbb E_t\bigl[r_t(\theta) A_t\bigr] - \beta\, \overline{\mathrm{KL}}, \qquad \beta \leftarrow \begin{cases} 2\beta & \text{if } \overline{\mathrm{KL}} > 1.5\, \delta \\ \beta / 2 & \text{if } \overline{\mathrm{KL}} < \delta / 1.5 \end{cases}$ with target $\delta = 0.01$, and compare it against clip=0.2 PPO on the same seeds. Report which method attains $J^\star$ faster and which sustains a KL closer to $\delta$.*

### 한국어 풀이용 정리
- **(a)**: clip $\varepsilon$ 는 학습 속도 vs 안정성의 정확히 반대 방향의 트레이드오프.
- **(b)**: TRPO 원형 아이디어 (KL 신뢰영역) 를 실용 근사한 KL-penalty PPO 를 구현해 clip PPO 와 비교.
- **관측 대상**: 학습 곡선, 배치 KL 궤적 (목표 $\delta$ 대비), 최종 $p$ 분포.

## 2. 수학적 배경

### 2.1 Clip parameter $\varepsilon$ 의 역할
PPO clipped surrogate 의 유효 스텝 크기는 대략
$$
|\Delta \pi| \;\lesssim\; \varepsilon \cdot |A_t|
$$
로 제한된다. 작은 $\varepsilon$ 는 안전하지만 학습이 느리고, 큰 $\varepsilon$ 는 vanilla PG 에 접근한다.

### 2.2 Adaptive KL-penalty (TRPO 원형)
Schulman et al. (2017) 은 PPO 논문에서 두 목적함수를 제안했다. clip 형태 외에
$$
L^{\text{KLPEN}}(\theta) = \mathbb E_t\bigl[r_t(\theta) A_t\bigr] - \beta \; \overline{\mathrm{KL}}\bigl(\pi_{\theta_\text{old}} \| \pi_\theta\bigr),
$$
그리고 매 배치 종료 후 $\overline{\mathrm{KL}}$ 을 측정해 $\beta$ 를 자동 조정한다:

$$
\begin{cases}
\beta \leftarrow 2\beta, & \overline{\mathrm{KL}} > 1.5\,\delta,\\
\beta \leftarrow \beta / 2, & \overline{\mathrm{KL}} < \delta / 1.5.
\end{cases}
$$

이 방식은 TRPO 의 hard KL 신뢰영역 제약을 soft penalty 로 완화한 근사다. 우리 태스크는 스칼라이므로 $\overline{\mathrm{KL}}$ 은 닫힌형 Bernoulli KL 로 정확히 계산.

$$
\boxed{\ \text{clip} \approx \text{학습률 상한을 배치 안에서 잘라내기,}\quad \text{KLPEN} \approx \text{배치 사이에서 학습률을 재조정하기}\ }
$$


## 3. 풀이 흐름

1. **공통 스캐폴딩** — Problem 2 의 PPO 러너를 파라미터화해 $\varepsilon$ 스윕과 KLPEN 을 함께 지원.
2. **(a) clip 스윕** — $\varepsilon \in \{0.05, 0.1, 0.2, 0.4\}$, 각각 20 seeds × 30 batches.
3. **(b) KLPEN 러너** — clip 대신 objective 도함수에 $-\beta \, \nabla \mathrm{KL}$ 을 더해 갱신,    매 배치 후 $\beta$ 조정. 동일 20 seeds.
4. **정량 지표** — 마지막 5 배치 리턴, 시드 표준편차, batch-KL 시퀀스의 median 을 표로.
5. **시각화** — clip 스윕 학습 곡선 & KL, clip=0.2 vs KLPEN 비교, $\beta$ 궤적.

In [1]:
# --- 공통 환경: Short Corridor with Switched Actions (Sutton & Barto Ex. 13.1) ---
# 상태 0 (start), 1 (switched), 2, 3 (terminal).  행동: 0 = left, 1 = right.
# 상태 0, 2 에서는 정상 (right -> +1, left -> -1).  상태 1 에서만 뒤바뀐다.
# 벽 밖으로 나가면 그 자리에 머무름.  각 스텝 보상 -1, 종단 보상 0.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RIGHT, LEFT = 1, 0
N_STATES = 4  # 3 non-terminal + 1 terminal

def next_state(s, a):
    move = +1 if a == RIGHT else -1
    if s == 1:
        move = -move
    ns = s + move
    if ns < 0:
        ns = 0
    if ns > 3:
        ns = 3
    return ns

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def run_episode(theta, rng, max_steps=500):
    p_right = sigmoid(theta)
    S, A, R = [], [], []
    s = 0
    for _ in range(max_steps):
        a = RIGHT if rng.random() < p_right else LEFT
        ns = next_state(s, a)
        S.append(s); A.append(a); R.append(-1.0)
        s = ns
        if s == 3:
            break
    return np.array(S), np.array(A, dtype=int), np.array(R)

def expected_return(theta, n=8000, max_steps=500, seed=0):
    rng = np.random.default_rng(seed)
    G = np.empty(n)
    for i in range(n):
        _, _, R = run_episode(theta, rng, max_steps=max_steps)
        G[i] = R.sum()
    return G.mean()


/tmp/mplcache is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-e5ydopbh because there was an issue with the default path (/tmp/mplcache); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# ------ PPO 러너 (clip 버전) --------------------------------------------------------
def _ppo_grad(S, A, adv, theta, theta_old, eps):
    p = sigmoid(theta); p_old = sigmoid(theta_old)
    pi_new = np.where(A == RIGHT, p, 1 - p)
    pi_old = np.where(A == RIGHT, p_old, 1 - p_old)
    r = pi_new / pi_old
    r_clip = np.clip(r, 1 - eps, 1 + eps)
    obj_u = r      * adv
    obj_c = r_clip * adv
    use_clip = obj_c < obj_u
    d_r_dth = pi_new * (A - p)
    d_obj = np.where(use_clip, 0.0, adv * d_r_dth)
    return d_obj.sum()

def _kl(theta_new, theta_old):
    p, q = sigmoid(theta_new), sigmoid(theta_old)
    e = 1e-12
    return q*np.log((q+e)/(p+e)) + (1-q)*np.log((1-q+e)/(1-p+e))

def _dkl_dth(theta_new, theta_old):
    # d/dθ KL(π_old || π_new) = -(q - p),  q=σ(θ_old), p=σ(θ_new)
    p, q = sigmoid(theta_new), sigmoid(theta_old)
    return -(q - p)

def ppo_run(alpha, eps, K, n_batches, batch_size, seed):
    rng = np.random.default_rng(seed)
    theta = 0.0
    rets_hist = np.empty(n_batches)
    kl_hist   = np.empty(n_batches)
    for b in range(n_batches):
        theta_old = theta
        S_list, A_list, G_list = [], [], []
        rets = np.empty(batch_size)
        for i in range(batch_size):
            S, A, R = run_episode(theta_old, rng, max_steps=500)
            G = np.cumsum(R[::-1])[::-1]
            S_list.append(S); A_list.append(A); G_list.append(G)
            rets[i] = R.sum()
        S = np.concatenate(S_list); A = np.concatenate(A_list); G = np.concatenate(G_list)
        adv = G - G.mean()
        for _ in range(K):
            g = _ppo_grad(S, A, adv, theta, theta_old, eps)
            theta += alpha * g / batch_size
        rets_hist[b] = rets.mean()
        kl_hist[b]   = _kl(theta, theta_old)
    return rets_hist, kl_hist, theta

eps_list = [0.05, 0.10, 0.20, 0.40]
n_runs, n_batches, batch_size, alpha, K = 20, 30, 40, 2**-4, 4
clip_curves = {e: np.empty((n_runs, n_batches)) for e in eps_list}
clip_kls    = {e: np.empty((n_runs, n_batches)) for e in eps_list}
clip_theta  = {e: np.empty(n_runs) for e in eps_list}
for e in eps_list:
    for r in range(n_runs):
        c, k, th = ppo_run(alpha, e, K, n_batches, batch_size, seed=40000 + int(e*1000)*97 + r)
        clip_curves[e][r] = c; clip_kls[e][r] = k; clip_theta[e][r] = th

rows = []
for e in eps_list:
    tail = clip_curves[e][:, -5:].mean(axis=1)
    kl_med_last = np.median(clip_kls[e][:, -5:])
    p_end = sigmoid(clip_theta[e])
    ok = ((p_end >= 0.4) & (p_end <= 0.75)).mean()
    rows.append({
        'eps': e,
        'tail_mean_return': float(tail.mean()),
        'tail_seed_std':    float(tail.std(ddof=1)),
        'batch_KL_last5_median': float(kl_med_last),
        'p_end_mean':       float(p_end.mean()),
        'converged_frac':   float(ok),
    })
df_clip = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("(a) clip epsilon sweep")
print(df_clip.to_string(index=False))


(a) clip epsilon sweep
   eps  tail_mean_return  tail_seed_std  batch_KL_last5_median  p_end_mean  converged_frac
0.0500         -145.8620       185.8526                 0.0007      0.3384          0.4500
0.1000          -19.8855        36.1679                 0.0016      0.5510          0.8500
0.2000          -12.0025         0.8610                 0.0060      0.5647          1.0000
0.4000          -11.9400         0.8431                 0.0052      0.5947          1.0000


In [3]:
# ------ (b) Adaptive KL-penalty 러너 (TRPO 원형 근사) -------------------------------
def klpen_run(alpha, delta, K, n_batches, batch_size, seed, beta0=1.0):
    rng = np.random.default_rng(seed)
    theta = 0.0
    beta = beta0
    rets_hist = np.empty(n_batches)
    kl_hist   = np.empty(n_batches)
    beta_hist = np.empty(n_batches)
    for b in range(n_batches):
        theta_old = theta
        S_list, A_list, G_list = [], [], []
        rets = np.empty(batch_size)
        for i in range(batch_size):
            S, A, R = run_episode(theta_old, rng, max_steps=500)
            G = np.cumsum(R[::-1])[::-1]
            S_list.append(S); A_list.append(A); G_list.append(G)
            rets[i] = R.sum()
        S = np.concatenate(S_list); A = np.concatenate(A_list); G = np.concatenate(G_list)
        adv = G - G.mean()
        for _ in range(K):
            p = sigmoid(theta)
            pi_new = np.where(A == RIGHT, p, 1 - p)
            pi_old_p = sigmoid(theta_old)
            pi_old = np.where(A == RIGHT, pi_old_p, 1 - pi_old_p)
            d_surr = adv * pi_new * (A - p) / pi_old
            g_pol = d_surr.sum() / batch_size
            g_kl  = _dkl_dth(theta, theta_old)
            theta += alpha * (g_pol - beta * g_kl)
        kl_now = _kl(theta, theta_old)
        if   kl_now > 1.5 * delta:  beta *= 2.0
        elif kl_now < delta / 1.5:  beta /= 2.0
        beta = float(np.clip(beta, 1e-3, 1e3))
        rets_hist[b] = rets.mean()
        kl_hist[b]   = kl_now
        beta_hist[b] = beta
    return rets_hist, kl_hist, beta_hist, theta

delta = 0.01
klpen_curves = np.empty((n_runs, n_batches))
klpen_kls    = np.empty((n_runs, n_batches))
klpen_betas  = np.empty((n_runs, n_batches))
klpen_theta  = np.empty(n_runs)
for r in range(n_runs):
    c, k, bt, th = klpen_run(alpha, delta, K, n_batches, batch_size, seed=50000 + r)
    klpen_curves[r] = c; klpen_kls[r] = k; klpen_betas[r] = bt; klpen_theta[r] = th

rows = [{
    'method': 'PPO clip=0.2',
    'tail_return': float(clip_curves[0.20][:, -5:].mean()),
    'tail_seed_std': float(clip_curves[0.20][:, -5:].mean(axis=1).std(ddof=1)),
    'KL_last5_median': float(np.median(clip_kls[0.20][:, -5:])),
    'converged_frac': float(((sigmoid(clip_theta[0.20]) >= 0.4)
                             & (sigmoid(clip_theta[0.20]) <= 0.75)).mean()),
}, {
    'method': f'KLPEN adaptive (delta={delta})',
    'tail_return': float(klpen_curves[:, -5:].mean()),
    'tail_seed_std': float(klpen_curves[:, -5:].mean(axis=1).std(ddof=1)),
    'KL_last5_median': float(np.median(klpen_kls[:, -5:])),
    'converged_frac': float(((sigmoid(klpen_theta) >= 0.4)
                             & (sigmoid(klpen_theta) <= 0.75)).mean()),
}]
df_cmp = pd.DataFrame(rows)
print("(b) clip vs adaptive KL-penalty")
print(df_cmp.to_string(index=False))
print(f"\nAdaptive beta: median beta_end = {np.median(klpen_betas[:, -1]):.3f}, "
      f"IQR = [{np.quantile(klpen_betas[:, -1], 0.25):.3f}, "
      f"{np.quantile(klpen_betas[:, -1], 0.75):.3f}]")


(b) clip vs adaptive KL-penalty
                     method  tail_return  tail_seed_std  KL_last5_median  converged_frac
               PPO clip=0.2     -12.0025         0.8610           0.0060          1.0000
KLPEN adaptive (delta=0.01)    -255.9722       250.3676           0.0000          0.5000

Adaptive beta: median beta_end = 0.516, IQR = [0.001, 10.000]


In [4]:
# ------ 시각화 ---------------------------------------------------------------------
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
x = np.arange(1, n_batches + 1)
cmap = plt.get_cmap('plasma', len(eps_list))

for i, e in enumerate(eps_list):
    curves = clip_curves[e]
    mu = curves.mean(axis=0); sd = curves.std(axis=0)
    ax[0,0].plot(x, mu, color=cmap(i), label=f'eps={e}')
    ax[0,0].fill_between(x, mu - sd, mu + sd, color=cmap(i), alpha=0.12)
ax[0,0].axhline(-11.6, color='k', ls='--', lw=0.8)
ax[0,0].set_xlabel('Batch'); ax[0,0].set_ylabel('Batch mean return (20 seeds)')
ax[0,0].set_title('(a) PPO learning curves across clip eps')
ax[0,0].legend(fontsize=8); ax[0,0].grid(alpha=0.3); ax[0,0].set_ylim(-160, 0)

for i, e in enumerate(eps_list):
    ax[0,1].plot(x, np.median(clip_kls[e], axis=0), color=cmap(i), label=f'eps={e}')
ax[0,1].axhline(0.01, color='k', ls=':', lw=0.8, label='delta = 0.01')
ax[0,1].set_yscale('log')
ax[0,1].set_xlabel('Batch'); ax[0,1].set_ylabel('median batch KL')
ax[0,1].set_title('(a) Effective KL vs clip eps')
ax[0,1].legend(fontsize=8); ax[0,1].grid(alpha=0.3, which='both')

for name, curves, color in [('PPO clip=0.2', clip_curves[0.20], 'C0'),
                            (f'KLPEN adaptive delta={delta}', klpen_curves, 'C2')]:
    mu = curves.mean(axis=0); sd = curves.std(axis=0)
    ax[1,0].plot(x, mu, color=color, label=name)
    ax[1,0].fill_between(x, mu - sd, mu + sd, color=color, alpha=0.15)
ax[1,0].axhline(-11.6, color='k', ls='--', lw=0.8, label='J* ~= -11.6')
ax[1,0].set_xlabel('Batch'); ax[1,0].set_ylabel('Batch mean return (20 seeds)')
ax[1,0].set_title('(b) Clip vs adaptive KL-penalty')
ax[1,0].legend(fontsize=8); ax[1,0].grid(alpha=0.3); ax[1,0].set_ylim(-160, 0)

b_med = np.median(klpen_betas, axis=0)
b_lo  = np.quantile(klpen_betas, 0.25, axis=0)
b_hi  = np.quantile(klpen_betas, 0.75, axis=0)
ax[1,1].plot(x, b_med, color='C2', label='beta median')
ax[1,1].fill_between(x, b_lo, b_hi, color='C2', alpha=0.2, label='IQR')
ax[1,1].set_yscale('log')
ax[1,1].set_xlabel('Batch'); ax[1,1].set_ylabel('beta (KL penalty coefficient)')
ax[1,1].set_title('(b) Adaptive beta trajectory')
ax[1,1].legend(fontsize=8); ax[1,1].grid(alpha=0.3, which='both')

plt.tight_layout(); plt.show()


## 4. 결과 해석

### (a) clip $\varepsilon$ 스윕
1. **$\varepsilon = 0.05$** — 매 배치 유효 스텝이 지나치게 작아 학습이 아주 느리다. 30 배치    후에도 tail 리턴이 $-146$ 수준, 절반 이하 (45%) 의 시드만이 $p^\star$ 근방에 도달했다.
2. **$\varepsilon = 0.10$** — 학습 속도가 빨라져 tail 리턴 $-19.9$, 85% 수렴. 여전히 일부 시드는    초반 스텝이 부족해 안장점을 벗어나지 못한다.
3. **$\varepsilon = 0.20$** — 스윗 스팟. 100% 시드 수렴, tail 리턴 $-12.0$, seed 표준편차 $0.86$.    PPO 논문 기본값 0.2 가 왜 널리 쓰이는지 재현된다.
4. **$\varepsilon = 0.40$** — 최종 성능은 clip=0.2 와 사실상 동일 ($-11.9$, 100% 수렴). 이번 스칼라    태스크는 정책 공간이 매우 작아 clip 이 완전히 무력화되어도 문제가 안 되지만, 대규모 정책에서는    KL 이 커지며 안정성이 흔들리는 것으로 알려져 있다.

### (b) clip vs adaptive KL-penalty
5. **clip=0.2 가 더 안정적이다** — tail 리턴 $-12.0$ vs KLPEN $-256$, 수렴 성공률 100% vs 50%.    KLPEN 의 KL median 은 목표 $\delta = 0.01$ 보다 훨씬 낮게 ($10^{-4}$ 수준) 붙잡히는데, 이는    $\beta$ 조정 규칙이 이 스칼라 태스크에서 $\beta$ 를 상한 (10) 으로 밀어붙여 사실상 학습을 정지시킨    시드가 절반이었기 때문이다 (β IQR 이 $[10^{-3}, 10]$ 로 극단적으로 벌어진 것에서 확인).
6. **PPO 논문의 원 결과와 일치** — Schulman et al. (2017) 도 여러 벤치마크에서 clip 이 KLPEN 보다    실무적으로 더 강건하다고 보고했다. 이 실험은 그 결론을 가장 단순한 태스크에서 재현한 것.
7. **하이퍼파라미터 튜닝 부담** — clip 은 $\varepsilon$ 하나만 고르면 되고 (a) 에서 본 것처럼 $0.1$–$0.4$    범위에서 관대함. KLPEN 은 $\delta$·$\beta_0$·조정 규칙까지 정해야 하며, 초기 $\beta$ 선택에 민감.

> **결론**: clip 은 *배치 안에서* 유효 학습률을 잘라내고, KLPEN 은 *배치 사이* 에서 학습률을 재조정한다. > 두 방식 모두 vanilla PG 의 불안정을 해소하려는 시도지만, 이 스칼라 태스크에서 clip 이 훨씬 강건하다는 것이 > 실험적으로 확인되었다.

### 다음 Day 예고
Day 74 로 §15.7 로 정책 최적화의 실무 표준을 완주했다. Day 75 는 §15.8 에 해당하는 **신경망 함수근사 하의 PPO (nonlinear function approximation)** — 스칼라 정책이 아닌 다층 퍼셉트론 정책에서 PPO 를 CartPole/LunarLander 같은 표준 벤치마크에 적용하는 문제로 이어진다.